主要实现功能：预处理在途货件

In [1]:
import pandas as pd
import datetime
import os

# 预处理发货单

In [2]:
# ----------------------------------读取货件文件 ----------------------------------
date_parm = '20261109'
run_date = pd.to_datetime(date_parm, format="%Y%m%d")
dates = pd.to_datetime('2026-11-09')
inbound_path = f'../src_data/在途货件/发货单清单{date_parm}.xlsx'
parms_path = f'../src_data/在途货件/预计到达.xlsx'
# yuzhan_path = f'../src_data/预占单/调拨单20250325.xlsx'
yuzhan_path = f'../src_data/预占单/未出库调拨单导出{date_parm}.xlsx'
INSPECTION_PATH = "../src_data/2026 查验货件登记表.xlsx"

# 发货单
inbound_total = pd.read_excel(inbound_path, sheet_name='system_发货单', usecols=['发货单号', '目的国家', '目的仓', '物流方式', '状态', '提货日期', '预计出运日期', '确认出运日期', '预计到仓日期', '实际到仓日期','货件号'], parse_dates=['提货日期', '预计出运日期', '确认出运日期', '实际到仓日期'])
inbound_detail = pd.read_excel(inbound_path, sheet_name='system_发货单明细', usecols=['发货单号', '仓库', 'ShipmentId','SKU', 'MSKU', '发货量', '差异量','预计出运日期'])

# 预计到达
parms_df = pd.read_excel(parms_path, usecols=['国家', '发货渠道', '预计时间'])
parms_df['发货渠道'] = parms_df['发货渠道'].str.strip()

# 预占单
yuzhan_detail = pd.read_excel(yuzhan_path, sheet_name='明细数据', usecols=['SKU', 'MSKU','调出数量', '调拨单号'])
yuzhan_total = pd.read_excel(yuzhan_path, sheet_name='单据数据', usecols=['调拨单号', '调出仓', '调入仓', '物流方式', '单据状态', '拣货状态', '出库状态', '预计出运日期'], parse_dates=['预计出运日期'])

d:\Programs\python\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
d:\Programs\python\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


In [3]:
yuzhan_detail

,SKU,MSKU,调出数量,调拨单号
0,SP001-407 White 37,SP001-407 White 37,40,TF2610290001
1,SP001-406 Black 42,SP001-406 Black 42,38,TF2610290001


In [4]:
yuzhan_total

,调拨单号,调出仓,调入仓,物流方式,单据状态,拣货状态,出库状态,预计出运日期
0,TF2610290001,FBT临时仓,SEEKWAY:US_FBA,快递,审核通过,未拣货,未出库,2026-11-08


In [5]:
inbound_detail['仓库'].unique()

array(['眼镜-广州仓', '手套-广州仓', 'HK_SIMARI_Limited:US_FBA', 'RIVMOUNT:US_FBA',
       'RIVMOUNT:CA_FBA', 'rivbos:US_FBA', 'SEEKWAY:US_FBA',
       'SEEKWAY:CA_FBA', '水鞋-广州仓', 'TK本地仓', 'SEEKWAY:JP_FBA',
       'rivbos:JP_FBA', 'HK_SIMARI_Limited:CA_FBA', 'FBT临时仓',
       'CIKERWEL:US_FBA', 'HK_SIMARI_Limited:JP_FBA', 'rivbos:UK_FBA',
       'rivbos:EU_FBA', 'SENWAYZON:EU_FBA', 'SIMARI:EU_FBA',
       'SENWAYZON:US_FBA'], dtype=object)

In [6]:
inbound_detail['仓库'] = inbound_detail['仓库'].str.replace('EU', 'DE')

In [7]:
inbound_detail['仓库'].unique()

array(['眼镜-广州仓', '手套-广州仓', 'HK_SIMARI_Limited:US_FBA', 'RIVMOUNT:US_FBA',
       'RIVMOUNT:CA_FBA', 'rivbos:US_FBA', 'SEEKWAY:US_FBA',
       'SEEKWAY:CA_FBA', '水鞋-广州仓', 'TK本地仓', 'SEEKWAY:JP_FBA',
       'rivbos:JP_FBA', 'HK_SIMARI_Limited:CA_FBA', 'FBT临时仓',
       'CIKERWEL:US_FBA', 'HK_SIMARI_Limited:JP_FBA', 'rivbos:UK_FBA',
       'rivbos:DE_FBA', 'SENWAYZON:DE_FBA', 'SIMARI:DE_FBA',
       'SENWAYZON:US_FBA'], dtype=object)

In [8]:
LogisticsMethod_map = pd.read_excel('../src_data/在途货件/预计到达.xlsx', sheet_name = '物流渠道映射')

# 将具体的渠道 映射为大渠道：快递、空运、海运
yuzhan_total = pd.merge(yuzhan_total, LogisticsMethod_map, how = 'left', on = '物流方式')
yuzhan_total.drop(columns=['物流方式'], inplace=True)
yuzhan_total.rename(columns={'大渠道': '物流方式'}, inplace=True)

# 确保每一个调拨单都能匹配上-物流方式
is_nan = yuzhan_total['物流方式'].isna()
if is_nan.sum() > 0:
    raise ValueError("缺失对于大渠道，请更新【物流渠道映射】子表")

In [9]:
yuzhan_total['物流方式'].unique()

array(['快递'], dtype=object)

In [10]:
# 删除 货件号 列中值为 'FBA18ZVZD2TK' 的行
inbound_total = inbound_total[inbound_total['货件号'] != 'FBA18ZVZD2TK']

# 删除 货件号 列中值为 'FBA18ZVZD2TK' 的行
inbound_detail = inbound_detail[inbound_detail['ShipmentId'] != 'FBA18ZVZD2TK']

# 删除不需要的列
inbound_total = inbound_total.drop(columns=['货件号'])

In [11]:
inbound_total

,发货单号,目的国家,目的仓,物流方式,状态,提货日期,预计出运日期,确认出运日期,预计到仓日期,实际到仓日期
0,DL2611099440,CN,眼镜-广州仓,国内快递,已出运,2026-11-06,2026-11-06,2026-11-09,2026-11-06,NaT
1,DL2611099439,CN,手套-广州仓,国内快递,已出运,2026-11-09,2026-11-09,2026-11-09,2026-11-09,NaT
2,DL2611099438,US,HK_SIMARI_Limited:US_FBA,原飞航_UHK03香港UPS美国红单大货包税价-特快/6000,已出运,2026-11-04,2026-11-04,2026-11-09,2026-11-22,NaT
3,DL2611099436,US,HK_SIMARI_Limited:US_FBA,原飞航_UHK03香港UPS美国红单大货包税价-特快/6000,已出运,2026-11-04,2026-11-06,2026-11-09,2026-11-13,NaT
4,DL2611099437,US,HK_SIMARI_Limited:US_FBA,原飞航_UHK03香港UPS美国红单大货包税价-特快/6000,已出运,2026-11-04,2026-11-04,2026-11-09,2026-11-11,NaT
...,...,...,...,...,...,...,...,...,...,...
824,DL2608138338,US,rivbos:US_FBA,速十美线海卡-美森正班船卡派,入库中,2026-08-12,2026-08-12,2026-08-13,2026-09-02,2026-09-14
825,DL2608138337,US,rivbos:US_FBA,速十美线海卡-美森正班船卡派,入库中,2026-08-12,2026-08-12,2026-08-13,2026-09-02,2026-09-12
826,DL2608138336,US,rivbos:US_FBA,速十美线海卡-美森正班船卡派,入库中,2026-08-12,2026-08-12,2026-08-13,2026-09-02,2026-09-06
827,DL2608138335,US,rivbos:US_FBA,速十_美国海派渠道（含税）15日限时达,入库中,2026-08-12,2026-08-12,2026-08-13,2026-09-02,2026-09-05


In [12]:
inbound_detail

,发货单号,ShipmentId,仓库,SKU,MSKU,发货量,差异量,预计出运日期
0,DL2611099440,NaN,眼镜-广州仓,833-Black Ice Blue Lens,833-Black Ice Blue Lens,40,40,2026-11-06
1,DL2611099440,NaN,眼镜-广州仓,833-BlackPink,833-Black&Pink,120,120,2026-11-06
2,DL2611099440,NaN,眼镜-广州仓,RB831- Full black,RB831- Full black,160,160,2026-11-06
3,DL2611099440,NaN,眼镜-广州仓,RB831-Black&Black Mirror Lens,3Y-Q321-AVWB,80,80,2026-11-06
4,DL2611099440,NaN,眼镜-广州仓,RB831-BlackGrey,A5-LU62-M9WH,360,360,2026-11-06
...,...,...,...,...,...,...,...,...
26869,DL2608138334,FBA19LJS8DNR,SEEKWAY:CA_FBA,SP001-408 Purple 40,SP001-408 Purple 40,3,0,2026-08-19
26870,DL2608138334,FBA19LJS8DNR,SEEKWAY:CA_FBA,SP001-417 Gradient Blue Pink 38,SP001-CA 417 Gradient Blue Pink 38,2,0,2026-08-19
26871,DL2608138334,FBA19LJS8DNR,SEEKWAY:CA_FBA,SP001-417 Gradient Blue Pink 39,SP001-CA 417 Gradient Blue Pink 39,1,0,2026-08-19
26872,DL2608138334,FBA19LJS8DNR,SEEKWAY:CA_FBA,SK001-781 Black 34-35,SK001-781 Black 34-35,35,0,2026-08-19


In [13]:
parms_df

,国家,发货渠道,预计时间
0,JP,FBA自提物流,7
1,US,德速_美国海卡限时达-美森正班快船-包税,40
2,CA,速十加拿大海空联运标准线,40
3,US,速十美线海卡-美森正班船卡派,40
4,US,速十_美国海派渠道（含税）15日限时达,40
...,...,...,...
152,US,原飞航_UHK03香港UPS美国红单大货包税价-特快/6000,10
153,DE,美通_欧洲空派普货标准线（卡派）,20
154,US,AGL物流空运,20
155,US,原飞航_USZ13大陆UPS美国红单大货包税价--特快/6000,10


In [14]:
yuzhan_detail

,SKU,MSKU,调出数量,调拨单号
0,SP001-407 White 37,SP001-407 White 37,40,TF2610290001
1,SP001-406 Black 42,SP001-406 Black 42,38,TF2610290001


In [15]:
yuzhan_total

,调拨单号,调出仓,调入仓,单据状态,拣货状态,出库状态,预计出运日期,物流方式
0,TF2610290001,FBT临时仓,SEEKWAY:US_FBA,审核通过,未拣货,未出库,2026-11-08,快递


In [16]:
# 待发货货件 ——todo:FBA货件待发货货件 可以迭代，处理不需要这部分数据
# daifahuo_total = pd.read_excel(f'../src_data/FBA货件待发货货件/FBA货件-{date_parm}.xlsx', sheet_name='FBA货件', usecols=['ShipmentID','仓库名称', '创建时间'])
# daifahuo_detail = pd.read_excel(f'../src_data/FBA货件待发货货件/FBA货件-{date_parm}.xlsx', sheet_name='FBA货件明细', usecols=['仓库名称', 'ShipmentID', 'SKU', 'MSKU', '申报量'])
daifahuo_total = pd.read_excel(f'../src_data/FBA货件待发货货件/FBA货件-2025.xlsx', sheet_name='FBA货件', usecols=['ShipmentID','仓库名称', '创建时间'])
daifahuo_detail = pd.read_excel(f'../src_data/FBA货件待发货货件/FBA货件-2025.xlsx', sheet_name='FBA货件明细', usecols=['仓库名称', 'ShipmentID', 'SKU', 'MSKU', '申报量'])

diaobo_total = pd.read_excel('../src_data/调拨单汇总表/调拨单汇总表 2024年6月.xlsx', skiprows=[0, 2], sheet_name='调拨单汇总表', usecols=['调入仓库', '货运方式', '货件编号', '提货状态', '实际出库日期', '后台货件状态', '站点'], parse_dates=['实际出库日期'])

In [17]:
# ----------------------------------货件信息预处理--------------------------------------
# inbound_total = inbound_total.query("~目的国家.str.contains('CN') and 状态.isin(['已出运', '入库中', '提货中'])", engine='python')
inbound_total = inbound_total.query("目的仓.str.contains('FBA') and 状态.isin(['已出运', '入库中', '提货中'])", engine='python')
# 删除不需要的列
inbound_total = inbound_total.drop(columns=['目的仓'])

In [18]:
inbound_total

,发货单号,目的国家,物流方式,状态,提货日期,预计出运日期,确认出运日期,预计到仓日期,实际到仓日期
2,DL2611099438,US,原飞航_UHK03香港UPS美国红单大货包税价-特快/6000,已出运,2026-11-04,2026-11-04,2026-11-09,2026-11-22,NaT
3,DL2611099436,US,原飞航_UHK03香港UPS美国红单大货包税价-特快/6000,已出运,2026-11-04,2026-11-06,2026-11-09,2026-11-13,NaT
4,DL2611099437,US,原飞航_UHK03香港UPS美国红单大货包税价-特快/6000,已出运,2026-11-04,2026-11-04,2026-11-09,2026-11-11,NaT
5,DL2611099435,US,原飞航_UHK03香港UPS美国红单大货包税价-特快/6000,已出运,2026-11-04,2026-11-06,2026-11-09,2026-11-13,NaT
6,DL2611099434,US,原飞航_UHK03香港UPS美国红单大货包税价-特快/6000,已出运,2026-11-04,2026-11-04,2026-11-09,2026-11-11,NaT
...,...,...,...,...,...,...,...,...,...
824,DL2608138338,US,速十美线海卡-美森正班船卡派,入库中,2026-08-12,2026-08-12,2026-08-13,2026-09-02,2026-09-14
825,DL2608138337,US,速十美线海卡-美森正班船卡派,入库中,2026-08-12,2026-08-12,2026-08-13,2026-09-02,2026-09-12
826,DL2608138336,US,速十美线海卡-美森正班船卡派,入库中,2026-08-12,2026-08-12,2026-08-13,2026-09-02,2026-09-06
827,DL2608138335,US,速十_美国海派渠道（含税）15日限时达,入库中,2026-08-12,2026-08-12,2026-08-13,2026-09-02,2026-09-05


In [19]:
# 获取处于`入库中`的MSKU的`接收天数`   计算货件处于“正在接收”的时间
# inbound_total['接收天数'] = (datetime.datetime.now() - inbound_total['实际到仓日期']).dt.days.fillna(0)
inbound_total['接收天数'] = (dates - inbound_total['实际到仓日期']).dt.days.fillna(0)

In [20]:
inbound_total['物流方式'] = inbound_total['物流方式'].str.strip()

# 将发货单_total与预计到达参数表合并，发货单为主表
inbound_ret = pd.merge(left=inbound_total, right=parms_df, left_on=['目的国家', '物流方式'], right_on=['国家', '发货渠道'], how='left')

# 确保每一个货件都能匹配上物流方式
is_nan = inbound_ret['发货渠道'].isna()
if is_nan.sum() > 0:
    # 创建保存路径
    excel_file_path = '../src_data/在途货件/物流方式缺失详情.xlsx'
    
    # 将包含缺失值的行保存到Excel文件
    inbound_ret[is_nan].to_excel(excel_file_path, index=False)
    raise ValueError(f"{os.path.abspath(excel_file_path)} 文件已生成，请更新预计到达表")

# 从后向前填充 '发货日期' 列
start_shipment = ['提货日期', '预计出运日期', '确认出运日期']
# 使用 fillna 方法填充缺失值.先把三个日期列倒序排好['确认出运日期', '预计出运日期', '提货日期']再用右边有值的单元格去填补左边空缺的单元格，最终只保留‘提货日期’列作为结果
inbound_ret['真-出运日期'] = inbound_ret[start_shipment[::-1]].fillna(method='bfill', axis=1)['提货日期']

# 检查 '真-出运日期' 列是否存在空值
if inbound_ret['真-出运日期'].isna().any():
    raise ValueError('存在出运时间为空，请检查发货单清单是否出现错误')


inbound_ret['预计到仓日期'] = inbound_ret['真-出运日期'] + pd.to_timedelta(inbound_ret.预计时间 + 1, unit='D')

C:\Users\gongy\AppData\Local\Temp\ipykernel_1452\2194802346.py:19: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  inbound_ret['真-出运日期'] = inbound_ret[start_shipment[::-1]].fillna(method='bfill', axis=1)['提货日期']


In [21]:
# TODO 增加了实际到仓时间
inbound_ret = inbound_ret[['发货单号', '物流方式', '实际到仓日期','预计到仓日期', '接收天数', '状态']]

# （发货单、预计到达）与发货单明细合表
merge_df = pd.merge(left=inbound_ret, right=inbound_detail, on=['发货单号'], how='left')

In [22]:
inbound_ret

,发货单号,物流方式,实际到仓日期,预计到仓日期,接收天数,状态
0,DL2611099438,原飞航_UHK03香港UPS美国红单大货包税价-特快/6000,NaT,2026-11-15,0.0,已出运
1,DL2611099436,原飞航_UHK03香港UPS美国红单大货包税价-特快/6000,NaT,2026-11-15,0.0,已出运
2,DL2611099437,原飞航_UHK03香港UPS美国红单大货包税价-特快/6000,NaT,2026-11-15,0.0,已出运
3,DL2611099435,原飞航_UHK03香港UPS美国红单大货包税价-特快/6000,NaT,2026-11-15,0.0,已出运
4,DL2611099434,原飞航_UHK03香港UPS美国红单大货包税价-特快/6000,NaT,2026-11-15,0.0,已出运
...,...,...,...,...,...,...
814,DL2608138338,速十美线海卡-美森正班船卡派,2026-09-14,2026-09-22,56.0,入库中
815,DL2608138337,速十美线海卡-美森正班船卡派,2026-09-12,2026-09-22,58.0,入库中
816,DL2608138336,速十美线海卡-美森正班船卡派,2026-09-06,2026-09-22,64.0,入库中
817,DL2608138335,速十_美国海派渠道（含税）15日限时达,2026-09-05,2026-09-22,65.0,入库中


In [23]:
merge_df

,发货单号,物流方式,实际到仓日期,预计到仓日期,接收天数,状态,ShipmentId,仓库,SKU,MSKU,发货量,差异量,预计出运日期
0,DL2611099438,原飞航_UHK03香港UPS美国红单大货包税价-特快/6000,NaT,2026-11-15,0.0,已出运,FBA19W2ZB1M2,HK_SIMARI_Limited:US_FBA,RG811 Black S,US-RG811-Black-S,12,12,2026-11-04
1,DL2611099438,原飞航_UHK03香港UPS美国红单大货包税价-特快/6000,NaT,2026-11-15,0.0,已出运,FBA19W2ZB1M2,HK_SIMARI_Limited:US_FBA,RG815 Black 2XL,US-RG815-Black-2XL,5,5,2026-11-04
2,DL2611099438,原飞航_UHK03香港UPS美国红单大货包税价-特快/6000,NaT,2026-11-15,0.0,已出运,FBA19W2ZB1M2,HK_SIMARI_Limited:US_FBA,RG810 Black L,US-RG810-Black-L,1,1,2026-11-04
3,DL2611099438,原飞航_UHK03香港UPS美国红单大货包税价-特快/6000,NaT,2026-11-15,0.0,已出运,FBA19W2ZB1M2,HK_SIMARI_Limited:US_FBA,SMRG933-Pink S,US-SG933-Pink-S,1,1,2026-11-04
4,DL2611099438,原飞航_UHK03香港UPS美国红单大货包税价-特快/6000,NaT,2026-11-15,0.0,已出运,FBA19W2ZB1M2,HK_SIMARI_Limited:US_FBA,RG811 Black L,US-RG811-Black-L,64,64,2026-11-04
...,...,...,...,...,...,...,...,...,...,...,...,...,...
26634,DL2608138334,速十加拿大海派快线,2026-10-06,2026-09-22,34.0,入库中,FBA19LJS8DNR,SEEKWAY:CA_FBA,SP001-408 Purple 40,SP001-408 Purple 40,3,0,2026-08-19
26635,DL2608138334,速十加拿大海派快线,2026-10-06,2026-09-22,34.0,入库中,FBA19LJS8DNR,SEEKWAY:CA_FBA,SP001-417 Gradient Blue Pink 38,SP001-CA 417 Gradient Blue Pink 38,2,0,2026-08-19
26636,DL2608138334,速十加拿大海派快线,2026-10-06,2026-09-22,34.0,入库中,FBA19LJS8DNR,SEEKWAY:CA_FBA,SP001-417 Gradient Blue Pink 39,SP001-CA 417 Gradient Blue Pink 39,1,0,2026-08-19
26637,DL2608138334,速十加拿大海派快线,2026-10-06,2026-09-22,34.0,入库中,FBA19LJS8DNR,SEEKWAY:CA_FBA,SK001-781 Black 34-35,SK001-781 Black 34-35,35,0,2026-08-19


In [24]:
# # TODO 用户自定义 初始化异常接收边界天数
# # 10天是用户自定义的异常接收边界天数
# boundary_day = 15

# # 使用 query 过滤数据 处于接收中并且接收天数小于等于15天的sku和差异量大于0的sku
# # 注意：接收天数为0时代表该MSKU还没到FBA
# boundary_msku = merge_df.query(f"0 < 接收天数 <= {boundary_day} and 差异量 != 0")

# # 重置 入库中 MSKU的预计到仓日期
# # boundary_msku['真-预计到仓日期'] = datetime.datetime.now() + pd.to_timedelta((boundary_day + 1 - boundary_msku['接收天数']), unit='D')
# boundary_msku['真-预计到仓日期'] = dates + pd.to_timedelta((boundary_day + 1 - boundary_msku['接收天数']), unit='D')

# # 更新原始 DataFrame
# merge_df.loc[boundary_msku.index, '预计到仓日期'] = boundary_msku['真-预计到仓日期']
# merge_df.loc[boundary_msku.index, '发货量'] = boundary_msku['差异量']

In [25]:
merge_df.query('MSKU == "SK002-Stripe Black 701 38-39"')

,发货单号,物流方式,实际到仓日期,预计到仓日期,接收天数,状态,ShipmentId,仓库,SKU,MSKU,发货量,差异量,预计出运日期
9786,DL2610199045,速十_美国海派渠道（含税）15日限时达,NaT,2026-11-24,0.0,已出运,FBA19T1JJ6RG,SEEKWAY:US_FBA,SK001-701 Stripe Black 38-39,SK002-Stripe Black 701 38-39,232,232,2026-10-14
9789,DL2610199046,速十美线海卡-美森正班船卡派,NaT,2026-11-24,0.0,已出运,FBA19T1K0RFQ,SEEKWAY:US_FBA,SK001-701 Stripe Black 38-39,SK002-Stripe Black 701 38-39,232,232,2026-10-14
9792,DL2610199044,速十美线海卡-美森正班船卡派,NaT,2026-11-24,0.0,已出运,FBA19T1H4VQL,SEEKWAY:US_FBA,SK001-701 Stripe Black 38-39,SK002-Stripe Black 701 38-39,174,174,2026-10-14
9795,DL2610199042,速十美线海卡-美森正班船卡派,NaT,2026-11-24,0.0,已出运,FBA19T1HPP2J,SEEKWAY:US_FBA,SK001-701 Stripe Black 38-39,SK002-Stripe Black 701 38-39,290,290,2026-10-14
9798,DL2610199043,速十美线海卡-美森正班船卡派,NaT,2026-11-24,0.0,已出运,FBA19T1BPMKL,SEEKWAY:US_FBA,SK001-701 Stripe Black 38-39,SK002-Stripe Black 701 38-39,290,290,2026-10-14
15399,DL2609288818,速十_美国海派渠道（含税）15日限时达,2026-10-22,2026-11-04,18.0,入库中,FBA19QZP4YQF,SEEKWAY:US_FBA,SK001-701 Stripe Black 38-39,SK002-Stripe Black 701 38-39,17,0,2026-09-24
15493,DL2609288817,速十_美国海派渠道（含税）15日限时达,2026-10-24,2026-11-04,16.0,入库中,FBA19QZXYQN0,SEEKWAY:US_FBA,SK001-701 Stripe Black 38-39,SK002-Stripe Black 701 38-39,17,0,2026-09-24
15587,DL2609288816,速十_美国海派渠道（含税）15日限时达,2026-10-22,2026-11-04,18.0,入库中,FBA19QZNDKQL,SEEKWAY:US_FBA,SK001-701 Stripe Black 38-39,SK002-Stripe Black 701 38-39,17,0,2026-09-24
15681,DL2609288815,速十_美国海派渠道（含税）15日限时达,2026-10-22,2026-11-04,18.0,入库中,FBA19QZM9N3L,SEEKWAY:US_FBA,SK001-701 Stripe Black 38-39,SK002-Stripe Black 701 38-39,17,0,2026-09-24
15775,DL2609288814,速十_美国海派渠道（含税）15日限时达,2026-10-22,2026-11-04,18.0,入库中,FBA19QZSFVT1,SEEKWAY:US_FBA,SK001-701 Stripe Black 38-39,SK002-Stripe Black 701 38-39,17,0,2026-09-24


In [26]:
# 在途货件
fba_inbound = merge_df.query(f"状态 != '入库中'")

# 接收中货件
fba_receive_detail = merge_df.query(f"状态 == '入库中'")

fba_receive_detail.rename(columns={'仓库': '店铺-站点'}, inplace=True)
fba_receive_detail['店铺-站点'] = fba_receive_detail['店铺-站点'].str.replace('_FBA', '')

fba_receive_detail['已收量'] = fba_receive_detail['发货量'] - fba_receive_detail['差异量']

fba_receive_total = fba_receive_detail.pivot_table(index=['店铺-站点', 'SKU', 'MSKU'], values=['发货量', '差异量', '已收量'], aggfunc=sum).reset_index()

C:\Users\gongy\AppData\Local\Temp\ipykernel_1452\1051739966.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  fba_receive_detail.rename(columns={'仓库': '店铺-站点'}, inplace=True)
C:\Users\gongy\AppData\Local\Temp\ipykernel_1452\1051739966.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  fba_receive_detail['店铺-站点'] = fba_receive_detail['店铺-站点'].str.replace('_FBA', '')
C:\Users\gongy\AppData\Local\Temp\ipykernel_1452\1051739966.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer]

In [27]:
fba_receive_detail

,发货单号,物流方式,实际到仓日期,预计到仓日期,接收天数,状态,ShipmentId,店铺-站点,SKU,MSKU,发货量,差异量,预计出运日期,已收量
2356,DL2611029355,FBA自提物流,2026-11-03,2026-11-09,6.0,入库中,FBA15GKZ53XZ,SEEKWAY:JP,WP002-208 Black 39,wp002-208 Black 39,19,0,2026-11-01,19
2357,DL2611029355,FBA自提物流,2026-11-03,2026-11-09,6.0,入库中,FBA15GKZ53XZ,SEEKWAY:JP,WP002-217 White 42,wp002-217 White 42,3,0,2026-11-01,3
2358,DL2611029355,FBA自提物流,2026-11-03,2026-11-09,6.0,入库中,FBA15GKZ53XZ,SEEKWAY:JP,WP002-215 Purplish blue 35,wp002-215 Purplish blue 35,2,0,2026-11-01,2
2359,DL2611029355,FBA自提物流,2026-11-03,2026-11-09,6.0,入库中,FBA15GKZ53XZ,SEEKWAY:JP,WP002-208 Black 38,wp002-208 Black 38,10,0,2026-11-01,10
2360,DL2611029355,FBA自提物流,2026-11-03,2026-11-09,6.0,入库中,FBA15GKZ53XZ,SEEKWAY:JP,WP002-238 White 41,JPWP002-238 White 41,2,0,2026-11-01,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
26634,DL2608138334,速十加拿大海派快线,2026-10-06,2026-09-22,34.0,入库中,FBA19LJS8DNR,SEEKWAY:CA,SP001-408 Purple 40,SP001-408 Purple 40,3,0,2026-08-19,3
26635,DL2608138334,速十加拿大海派快线,2026-10-06,2026-09-22,34.0,入库中,FBA19LJS8DNR,SEEKWAY:CA,SP001-417 Gradient Blue Pink 38,SP001-CA 417 Gradient Blue Pink 38,2,0,2026-08-19,2
26636,DL2608138334,速十加拿大海派快线,2026-10-06,2026-09-22,34.0,入库中,FBA19LJS8DNR,SEEKWAY:CA,SP001-417 Gradient Blue Pink 39,SP001-CA 417 Gradient Blue Pink 39,1,0,2026-08-19,1
26637,DL2608138334,速十加拿大海派快线,2026-10-06,2026-09-22,34.0,入库中,FBA19LJS8DNR,SEEKWAY:CA,SK001-781 Black 34-35,SK001-781 Black 34-35,35,0,2026-08-19,35


# 处理预占单据

In [28]:
# 筛选调拨数量大于0的SKU
yuzhan_detail = yuzhan_detail[yuzhan_detail['调出数量'] > 0]

In [29]:
yuzhan_detail

,SKU,MSKU,调出数量,调拨单号
0,SP001-407 White 37,SP001-407 White 37,40,TF2610290001
1,SP001-406 Black 42,SP001-406 Black 42,38,TF2610290001


In [30]:
# 取调入仓为FBA的调拨单，且单据状态为审核通过，出库状态为未出库
yuzhan_total = yuzhan_total.query("调入仓.str.contains('FBA') and 单据状态 == '审核通过' and 出库状态 == '未出库'")
yuzhan_total

,调拨单号,调出仓,调入仓,单据状态,拣货状态,出库状态,预计出运日期,物流方式
0,TF2610290001,FBT临时仓,SEEKWAY:US_FBA,审核通过,未拣货,未出库,2026-11-08,快递


In [31]:
# 获取当前日期
# yuzhan_total['预占天数'] = (datetime.datetime.now() - yuzhan_total['预计出运日期']).dt.days.fillna(0)
yuzhan_total['预占天数'] = (dates - yuzhan_total['预计出运日期']).dt.days.fillna(0)

# TODO 用户自定义 过滤异常数据(例如：海外仓调拨需求在2023-10月发起，但是可能会在2023-11月还没操作)
exception_days = 30
yuzhan_total_true = yuzhan_total.query(f"预占天数 <= {exception_days}")
yuzhan_total_flase = yuzhan_total.query(f"预占天数 > {exception_days}")

yuzhan_total_true ['国家'] = yuzhan_total_true .调入仓.str.extract('.*:(.{2})_*')
yuzhan_total_true ['国家'] = yuzhan_total_true ['国家'].apply(lambda x: 'DE' if x == 'EU' else x)

In [32]:
yuzhan_total

,调拨单号,调出仓,调入仓,单据状态,拣货状态,出库状态,预计出运日期,物流方式,预占天数
0,TF2610290001,FBT临时仓,SEEKWAY:US_FBA,审核通过,未拣货,未出库,2026-11-08,快递,1


In [33]:
# 预占（调拨单-单据数据）与 预计到达表合并
yuzhan_total_true  = pd.merge(left=yuzhan_total_true , right=parms_df, left_on=['物流方式', '国家'], right_on=['发货渠道', '国家'], how='left')

is_nan = yuzhan_total_true ['发货渠道'].isna()
if is_nan.sum() > 0:
    # 创建保存路径
    excel_file_path = '../src_data/在途货件/物流方式缺失详情.xlsx'
    
    # 将包含缺失值的行保存到Excel文件
    yuzhan_total_true [is_nan].to_excel(excel_file_path, index=False)
    raise ValueError(f"{os.path.abspath(excel_file_path)} 文件已生成，请更新预计到达表")

In [34]:
yuzhan_total_true

,调拨单号,调出仓,调入仓,单据状态,拣货状态,出库状态,预计出运日期,物流方式,预占天数,国家,发货渠道,预计时间
0,TF2610290001,FBT临时仓,SEEKWAY:US_FBA,审核通过,未拣货,未出库,2026-11-08,快递,1,US,快递,10


In [35]:
yuzhan_total_true ['预计到仓日期'] = datetime.datetime.now()  # 获取当前的本地日期和时间

# 计算到库天数
yuzhan_total_true ['到库天数'] = (yuzhan_total_true ['预占天数'] > 0) * exception_days  - yuzhan_total_true ['预占天数'] + yuzhan_total_true ['预计时间']
yuzhan_total_true ['预计到仓日期'] = pd.to_datetime(yuzhan_total_true ['预计到仓日期']) + pd.to_timedelta(yuzhan_total_true ['到库天数'], unit='D')
# 预占总与预占详情合并
yuzhan_total_true = pd.merge(left=yuzhan_total_true , right=yuzhan_detail, on=['调拨单号'])

In [36]:
yuzhan_total_true ['预占天数'] > 0

0    True
1    True
Name: 预占天数, dtype: bool

In [37]:
yuzhan_total_true

,调拨单号,调出仓,调入仓,单据状态,拣货状态,出库状态,预计出运日期,物流方式,预占天数,国家,发货渠道,预计时间,预计到仓日期,到库天数,SKU,MSKU,调出数量
0,TF2610290001,FBT临时仓,SEEKWAY:US_FBA,审核通过,未拣货,未出库,2026-11-08,快递,1,US,快递,10,2026-12-18 17:14:19.196645,39,SP001-407 White 37,SP001-407 White 37,40
1,TF2610290001,FBT临时仓,SEEKWAY:US_FBA,审核通过,未拣货,未出库,2026-11-08,快递,1,US,快递,10,2026-12-18 17:14:19.196645,39,SP001-406 Black 42,SP001-406 Black 42,38


In [38]:
datetime.datetime.now()

datetime.datetime(2026, 11, 9, 17, 14, 19, 230498)

#### 将 调出仓 为“FBT临时仓”的数据拉出来

In [39]:
FBT_df = yuzhan_total_true.query("调出仓.str.contains('FBT') and 单据状态 == '审核通过' and 出库状态 == '未出库'")
# 根据 '调出仓', 'SKU', 'MSKU'分组，聚合 '调出数量'
FBT_df = FBT_df.groupby(['调入仓', 'SKU', 'MSKU'])['调出数量'].sum().reset_index()
# 将 调入仓 重命名为 店铺-站点
FBT_df.rename(columns={'调入仓': '店铺-站点', '调出数量': 'FBT->FBA的预占数','SKU': '积加SKU'}, inplace=True)
# 将 店铺-站点 值中的 _FBA 替换为空字符串
FBT_df['店铺-站点'] = FBT_df['店铺-站点'].str.replace('_FBA', '')

In [40]:
FBT_df.columns

Index(['店铺-站点', '积加SKU', 'MSKU', 'FBT->FBA的预占数'], dtype='object')

In [41]:
FBT_df

,店铺-站点,积加SKU,MSKU,FBT->FBA的预占数
0,SEEKWAY:US,SP001-406 Black 42,SP001-406 Black 42,38
1,SEEKWAY:US,SP001-407 White 37,SP001-407 White 37,40


# 处理待发货货件

In [42]:
diaobo_total.drop_duplicates(subset=['调入仓库', '货件编号'], inplace=True)

daifahuo_fba = pd.merge(left=daifahuo_total, right=diaobo_total, how='left', left_on=['仓库名称', 'ShipmentID'], right_on=['调入仓库', '货件编号']).drop(columns=['调入仓库', '货件编号'])


daifahuo_fba = pd.merge(left=daifahuo_fba, right=parms_df, left_on=['站点', '货运方式'], right_on=['国家', '发货渠道'], how='left')

In [43]:
diaobo_total.query('货件编号 == "FBA184242FTD"')

,站点,调入仓库,货运方式,货件编号,提货状态,实际出库日期,后台货件状态
13,US,SEEKWAY:US_FBA,海运,FBA184242FTD,已出运,2024-06-18,NaN


In [44]:
# TODO 需要复核未提走的订单
daifahuo_fba.query("提货状态 == '已提走'", inplace=True)

In [45]:
# 确保每一个货件都能匹配上物流方式
is_nan = daifahuo_fba['发货渠道'].isna()
if is_nan.sum() > 0:
    # 创建保存路径
    excel_file_path = '../src_data/在途货件/物流方式缺失详情.xlsx'
    
    # 将包含缺失值的行保存到Excel文件
    daifahuo_fba[is_nan].to_excel(excel_file_path, index=False)
    raise ValueError(f"{os.path.abspath(excel_file_path)} 文件已生成，请更新预计到达表")

daifahuo_fba['预计到仓日期'] = daifahuo_fba['实际出库日期'] + pd.to_timedelta(daifahuo_fba.预计时间 + 2, unit='D')

daifahuo_fba = pd.merge(left=daifahuo_fba, right=daifahuo_detail, on=['仓库名称', 'ShipmentID'])

In [46]:
# 将货件状态为“入库中”的货件与预占单、待发货货件表concat合并

stander_titles = ['仓库', 'SKU', 'MSKU','发货量', '预计到仓日期', '预计出运日期','状态', '物流方式', 'ShipmentId']
df1 = fba_inbound[stander_titles]

# TODO：现在没有预占数据了，考虑不要这部分数据（ 20251215-又有了，未拣货数据算做预占数据且为在途数）
df2 = yuzhan_total_true[['调入仓', 'SKU', 'MSKU', '调出数量', '预计到仓日期', '预计出运日期', '出库状态', '发货渠道']]
df2['ShipmentId'] = '预占单无ShipmentId'
df2.columns = stander_titles

# TODO：现在没有调拨单未提走的货件数据了，考虑不要这部分数据
df3 = daifahuo_fba[['仓库名称', 'SKU', 'MSKU','申报量', '预计到仓日期', '创建时间','提货状态', '发货渠道', 'ShipmentID']]
df3.columns = stander_titles
# 在途 = FBA在途 + 预占 + 接收中
ret_df = pd.concat(objs=[df1, df2, df3], axis=0)

C:\Users\gongy\AppData\Local\Temp\ipykernel_1452\1945782573.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df2['ShipmentId'] = '预占单无ShipmentId'


In [47]:
df1.query('MSKU =="004-Wayfarer Black"')

,仓库,SKU,MSKU,发货量,预计到仓日期,预计出运日期,状态,物流方式,ShipmentId
5082,rivbos:US_FBA,RBK004-2 W Black,004-Wayfarer Black,150,2026-12-03,2026-10-23,已出运,速十_美国海派渠道（含税）15日限时达,FBA19V455C00
5121,rivbos:US_FBA,RBK004-2 W Black,004-Wayfarer Black,150,2026-12-03,2026-10-23,已出运,速十美线海卡-美森正班船卡派,FBA19V41F043
5160,rivbos:US_FBA,RBK004-2 W Black,004-Wayfarer Black,150,2026-12-03,2026-10-23,已出运,速十_美国海派渠道（含税）15日限时达,FBA19V3YDSZ7
5199,rivbos:US_FBA,RBK004-2 W Black,004-Wayfarer Black,150,2026-12-03,2026-10-23,已出运,速十美线海卡-美森正班船卡派,FBA19V4410M7
5243,rivbos:US_FBA,RBK004-2 W Black,004-Wayfarer Black,150,2026-12-03,2026-10-23,已出运,速十美线海卡-美森正班船卡派,FBA19V45H81Q
8363,rivbos:US_FBA,RBK004-2 W Black,004-Wayfarer Black,180,2026-11-26,2026-10-21,已出运,速十_美国海派渠道（含税）15日限时达,FBA19T6RR4WF
8409,rivbos:US_FBA,RBK004-2 W Black,004-Wayfarer Black,180,2026-11-26,2026-10-21,已出运,速十_美国海派渠道（含税）15日限时达,FBA19T6QXHXK
8450,rivbos:US_FBA,RBK004-2 W Black,004-Wayfarer Black,180,2026-11-26,2026-10-16,已出运,速十美线海卡-美森正班船卡派,FBA19T6P6LW8
8496,rivbos:US_FBA,RBK004-2 W Black,004-Wayfarer Black,180,2026-11-26,2026-10-16,已出运,速十美线海卡-美森正班船卡派,FBA19T6T4VM6
8542,rivbos:US_FBA,RBK004-2 W Black,004-Wayfarer Black,180,2026-11-26,2026-10-16,已出运,速十美线海卡-美森正班船卡派,FBA19T6SZK0M


### 开始处理查验货件部分

In [48]:
def parse_date_series(series, actual_only=False):
    """从 Excel 日期列中提取日期。

    物流追踪表里会出现「预计2026-05-17」这类文本。
    这种预计节点还是当作真实物流时间使用，将预计去掉。
   
    """
    text = series.astype("string")
    extracted = text.str.extract(r"(\d{4})[-/](\d{1,2})[-/](\d{1,2})")
    date_text = extracted[0] + "-" + extracted[1].str.zfill(2) + "-" + extracted[2].str.zfill(2)
    parsed = pd.to_datetime(date_text, format="%Y-%m-%d", errors="coerce")
    # if actual_only:
    #     parsed = parsed.mask(text.str.contains("预计", na=False))  # 如果只要真实节点，遇到带“预计”的文本就当作缺失处理；但实际测试中发现有些追踪表里节点时间前面带了“预计”但其实是准确的真实节点，所以先不区分，统一提取日期部分作为实际时间。
    return parsed

In [49]:
def read_inspection_rules():
    """读取查验货件规则。

    返回两个结果：
    - excluded：不纳入库存计算的货件号集合。
    - override_eta：纳入库存计算的货件号及指定签收日期。
    """

    inspection = pd.read_excel(
        INSPECTION_PATH,
        sheet_name="汇总",
        usecols=["FBA ID", "是否纳入库存计算", "纳入库存计算后签收日期"],
    )
    
    # FBA ID 单元格可能夹带备注或运单号，只抽取 FBA 开头的货件号。
    inspection["货件号"] = inspection["FBA ID"].astype(str).str.extract(r"(FBA[0-9A-Z]+)", expand=False)

    # 不纳入库存计算：从发货单主表剔除，后续不会进入在途库存。
    excluded = set(
        inspection.loc[
            inspection["是否纳入库存计算"].astype(str).str.strip().eq("否"),
            "货件号",
        ].dropna()
    )

    # 纳入库存计算：保留货件，但预计到仓日期直接使用登记表指定日期。
    included = inspection[inspection["是否纳入库存计算"].astype(str).str.strip().eq("是")].copy()
    included["查验预计到仓日期"] = parse_date_series(included["纳入库存计算后签收日期"])
    if included["查验预计到仓日期"].isna().any():
        raise ValueError("查验货件登记表中存在纳入库存计算但签收日期为空的货件，请检查")

    # 删除货件号为空的记录，货件号去重，把货件号设置成索引，取出一列。
    override_eta = included.dropna(subset=["货件号"]).drop_duplicates("货件号").set_index("货件号")["查验预计到仓日期"]
    # print("纳入库存计算的货件号及指定签收日期：", override_eta)
    
    print("查验货件中不纳入库存计算的货件数：", len(excluded))
    print("查验货件中纳入库存计算的货件数：",len(override_eta))
    # print(override_eta)
    
    return excluded, override_eta

In [50]:
# inspection_excluded不纳入库存计算的货件列表集合，inspection_eta纳入库存计算的货件及指定签收日期
inspection_excluded, inspection_eta = read_inspection_rules()

# 保留不纳入库存计算的查验数据
inspection_excluded_df = ret_df[ret_df["ShipmentId"].isin(inspection_excluded)]
inspection_excluded_df = inspection_excluded_df[['仓库', 'SKU', 'MSKU', '发货量', '预计出运日期', '状态', '物流方式', 'ShipmentId']]

inspection_excluded_total = (
        inspection_excluded_df.pivot_table(
            index=["仓库", "SKU", "MSKU"],
            values=["发货量"],
            aggfunc="sum",
        )
        .reset_index()
    )

# 查验表：不纳入库存的货件直接剔除；纳入库存的货件保留并覆盖到仓日期。
ret_df = ret_df[~ret_df["ShipmentId"].isin(inspection_excluded)].copy()

# 修改查验货件的预计到仓日期
for shipment_id, eta in inspection_eta.items():
    ret_df.loc[ret_df["ShipmentId"] == shipment_id, "预计到仓日期"] = eta

查验货件中不纳入库存计算的货件数： 40
查验货件中纳入库存计算的货件数： 81


In [51]:
# 确保列是日期时间类型
ret_df['预计到仓日期'] = pd.to_datetime(ret_df['预计到仓日期'], errors='coerce')


# 将"仓库"列中所有的 'EU'（欧盟）替换为 'DE'（德国），其他值不变。
ret_df ['仓库'] = ret_df ['仓库'].apply(lambda x: 'DE' if x == 'EU' else x)

### 处理到仓日期早于当天日期部分

In [52]:
# 如果预计到仓日期早于运行日期，说明实际还未到仓，则预计到仓日期加7天，若+7天还是早于运行日期，则先不计入在途。
future_mask = ret_df["预计到仓日期"] < run_date
# 将这部分数据输出到异常表，方便后续检查数据质量。
if future_mask.any():
    future_records_detail = ret_df[future_mask].copy()
    future_records_detail["预计到仓日期+7天"] = future_records_detail["预计到仓日期"] + pd.Timedelta(days=7)
    # 新增列，说明是否计入在途，预计到仓日期+7天正常后则计入在途，否则不计入在途。
    future_records_detail["是否计入在途"] = (
        future_records_detail["预计到仓日期+7天"]    # 如果 预计到仓日期+7天 >= run_date，则该行 是否计入在途 为 True，否则为 False。
        .ge(run_date)
        .map({True: "是", False: "否"})
    )
    
    # future_records_detail.to_excel(f"../src_data/在途货件/{date_parm} 在途中货件预计到仓日期早于运行日期的记录.xlsx", index=False)
    future_records_total = (
        future_records_detail.pivot_table(
            index=["仓库", "SKU", "MSKU","是否计入在途"],
            values=["发货量"],
            aggfunc="sum",
        )
        .reset_index()
    )
else:
    print(f"{date_parm} 在途中货件预计到仓日期早于运行日期的记录为空，无需输出异常表。\n")

# 替换预计到仓日期早于运行日期的货件的预计到仓日期加7天，若+7天后仍然早于运行日期，则先不计入在途。
ret_df.loc[future_mask, "预计到仓日期"] = ret_df.loc[future_mask, "预计到仓日期"] + pd.Timedelta(days=7)


# 输出在途库存

In [53]:
# format date
ret_df['预计到仓日期'] = ret_df['预计到仓日期'].dt.strftime('%Y-%m-%d')

with pd.ExcelWriter(path=f'../src_data/在途货件/在途货件{date_parm}.xlsx') as writer:
    ret_df.to_excel(writer, sheet_name='在途货件-正常', index=False)
    fba_receive_total.to_excel(writer, sheet_name='接收中-汇总', index=False)
    fba_receive_detail.to_excel(writer, sheet_name='接收中-货件详情', index=False)
    future_records_detail.to_excel(writer, sheet_name='预计到仓日期早于当天日期的记录', index=False)
    future_records_total.to_excel(writer, sheet_name='预计到仓日期早于当天日期的记录-汇总', index=False)
    inspection_excluded_df.to_excel(writer, sheet_name='不计入在途的查验货件详情', index = False)
    inspection_excluded_total.to_excel(writer, sheet_name='不计入在途的查验货件MSKU汇总', index = False)
    yuzhan_total_flase.to_excel(writer, sheet_name=f'预占超过{exception_days}天', index=False)
    
    FBT_df.to_excel(writer, sheet_name='FBT-FBA的预占数汇总', index=False)

In [54]:
print(ret_df['预计到仓日期'].dtype)

object
